# Limpieza de `DATOS_TEC` (base para el recomendador)

Salidas en `data/processed/` (ignoradas por git; cifrar con `scripts/dvc_encrypt.sh`):

- `DATOS_TEC_clean.csv`: una fila por producto, `codigoproducto` como llave.
  - Propiedades que venían como texto (`> 40`, `< 0.55`, `310 - 330`) quedan como **un solo número**
    (el límite, o el punto medio si es rango) y, cuando hay mezcla, una columna `<propiedad>_tipo` con `=`, `>`, `<` o `rango`.
  - `textura` se separa en banderas `textura_suave`, `textura_adherente`, `textura_fluida`, `textura_viscosa`.
  - `grado_nlgi_por_penetracion`: grado NLGI calculado con la penetración (distingue 00 = -1 y 000 = -2, que el grado declarado junta en 0).
  - Texto en minúsculas y sin acentos.
- `DATOS_TEC_revisar.csv`: productos con datos contradictorios para confirmar con Interlub (no se borran ni se corrigen aquí).

In [ ]:
import re
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)

RAW = "../data/raw/DATOS_TEC.xlsx"
OUT = "../data/processed/DATOS_TEC_clean.csv"
REVIEW = "../data/processed/DATOS_TEC_revisar.csv"
MIN_DATOS = 5  # columns with fewer non-null values are dropped

df = pd.read_excel(RAW)
df.head()

In [ ]:
# 1. Baseline: size, dtypes and heaviest missing columns (reference for every later step)
print(df.shape)
print(df.dtypes.value_counts())
df.isna().mean().mul(100).round(1).sort_values(ascending=False).head(10)

In [ ]:
# 2. Snake_case ascii names (no accents, °, %, dots) and drop the Excel index
def clean_name(c):
    c = c.replace("%", "pct")
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_")


df = df.drop(columns="Unnamed: 0", errors="ignore")
names = pd.DataFrame({"before": df.columns, "after": [clean_name(c) for c in df.columns]})
assert names["after"].is_unique
df.columns = names["after"].tolist()
names[names.before != names.after]

In [ ]:
# 3. Duplicates and fully empty rows/columns; also repeated specs under different product codes
shape_0 = df.shape
df = df.drop_duplicates().dropna(how="all").dropna(axis=1, how="all")
assert df["codigoproducto"].is_unique
print(shape_0, "->", df.shape)
print("same specs, different code:", df.drop(columns="codigoproducto").duplicated().sum())

In [ ]:
# 4a. Text: trim/collapse spaces, lowercase and strip accents (product code untouched)
def norm(s):
    s = s.str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
    return s.str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")


text_cols = df.select_dtypes(exclude="number").columns.drop("codigoproducto")
df[text_cols] = df[text_cols].apply(norm)

# Oils carrying grease-only properties (texture, lumps, cone penetration): kept, listed for review in step 12
grease_only = ["penetracion_de_cono_a_25c_0_1mm", "textura", "textura_valor", "deteccion_de_grumo_valor"]
aceite_tipo_grasa = (df["familia"] == "aceite") & df[grease_only].notna().any(axis=1)

In [ ]:
# 4b. Homologate categories
df["aceite_base"] = df["aceite_base"].replace({"semi-sintetico": "semisintetico"})

# color: "ligero" == "claro" (and noun first), gender, hyphen
df["color"] = (
    df["color"]
    .str.replace("ligero verde", "verde claro")
    .str.replace("ligero", "claro")
    .str.replace("-", " ")
    .replace({"roja": "rojo"})
)

# QC values: drop the filler suffix " estandar" (cut to 20 chars at the source: " estand") and non-values
suffix = r"\s+estand(ar)?$"
df["color_visual_valor"] = (
    df["color_visual_valor"]
    .str.replace(suffix, "", regex=True)
    .replace({"gris obscuro": "gris oscuro", "color igual a": np.nan})
)
df["textura_valor"] = df["textura_valor"].str.replace(suffix, "", regex=True).replace({"ligera fibra": "fibrosa"})

# inspeccion_visual_valor is appearance; texture/lump words belong to other columns
misplaced = df["inspeccion_visual_valor"].isin(["sin grumo", "adherente", "fibrosa"])
print("moved out of inspeccion_visual_valor:", df.loc[misplaced, ["codigoproducto", "inspeccion_visual_valor"]].values.tolist())
df.loc[misplaced, "inspeccion_visual_valor"] = np.nan

# textura combines several properties ("fluido viscoso adherente"): one flag per property, NA when unknown
tokens = {"suave": "suave", "adherente": "adherente", "fluido": "fluida", "fluida": "fluida", "viscoso": "viscosa", "viscosa": "viscosa"}
words = df["textura"].str.split()
assert set(words.dropna().explode()) <= set(tokens), "unmapped texture word"
pos = df.columns.get_loc("textura")
for i, t in enumerate(dict.fromkeys(tokens.values())):
    flag = words.map(lambda w, t=t: any(tokens[x] == t for x in w) if isinstance(w, list) else pd.NA)
    df.insert(pos + i, "textura_" + t, flag.astype("boolean"))
df = df.drop(columns="textura")

for c in ["familia", "aceite_base", "espesante", "color", "color_visual_valor", "textura_valor", "inspeccion_visual_valor"]:
    print(c, df[c].value_counts().to_dict())
df[["textura_" + t for t in dict.fromkeys(tokens.values())]].apply(lambda s: s.value_counts(dropna=False)).T

In [ ]:
# 5. Text columns that are really numeric: check which entries are not plain numbers (bounds, ranges, placeholders)
plain = r"^-?\d+(\.\d+)?$"
numeric_like = r"^[<>=\s]*\d+(\.\d+)?(\s*-\s*\d+(\.\d+)?)?\s*(kg)?$"
text_cols = df.select_dtypes(exclude=["number", "boolean"]).columns
num_cols = [c for c in text_cols if df[c].dropna().str.match(numeric_like).mean() > 0.9]
rows = []
for c in num_cols:
    s = df[c].dropna()
    odd = s[~s.str.match(plain)]
    rows.append({"column": c, "n_plain": len(s) - len(odd), "n_odd": len(odd), "examples": list(odd.unique()[:4])})
pd.DataFrame(rows).set_index("column")

In [ ]:
# 6. One float per property: x -> x ("="), >x -> x (">"), <x -> x ("<"), a - b -> midpoint ("rango").
#    A "<col>_tipo" column is added only when a column mixes kinds. Unparseable entries are listed and set to NaN.
pattern = re.compile(r"(?:([<>])=?)?\s*(\d+(?:\.\d+)?)(?:\s*-\s*(\d+(?:\.\d+)?))?\s*(?:kg)?")


def parse(v):
    m = pattern.fullmatch(v) if isinstance(v, str) else None
    if not m:
        return np.nan, None
    op, lo, hi = m.groups()
    if hi:
        return (float(lo) + float(hi)) / 2, "rango"
    return float(lo), op or "="


not_numeric, kinds = [], {}
for c in num_cols:
    parsed = pd.DataFrame(df[c].map(parse).tolist(), index=df.index, columns=["value", "tipo"])
    lost = df[c].notna() & parsed["value"].isna()
    not_numeric += [(c, df.at[i, "codigoproducto"], df.at[i, c]) for i in df.index[lost]]
    kinds[c] = parsed["tipo"].value_counts()
    df[c] = parsed["value"].astype(float)
    if parsed["tipo"].nunique() > 1:
        df.insert(df.columns.get_loc(c) + 1, c + "_tipo", parsed["tipo"])

# every extrema_presion value is ">= N kg": it is a minimum in kg
df = df.rename(columns={"extrema_presion_valor": "extrema_presion_min_kg"})

print("not numeric -> NaN:", not_numeric)
pd.DataFrame(kinds).T.fillna(0).astype(int)

In [ ]:
# 7. Placeholders for "no limit": all-9s (999..999999) or 0 in *_max, 0 in *_min and *_estandar (kept only where a real 0 is possible)
real_zero = ("color_numerico", "estabilidad_mecanica", "lavado_por_agua", "tan_mg_koh_g")
nines = [999, 9999, 99999, 999999]
limits = [c for c in df.columns if not c.startswith("temperatura")]
maxs = [c for c in limits if c.endswith("_max")]
zeros = [c for c in limits if c.endswith("_min") or (c.endswith("_estandar") and not c.startswith(real_zero))]

na_before = df.isna().sum()
df[maxs] = df[maxs].mask(df[maxs].isin(nines) | (df[maxs] == 0))
df[zeros] = df[zeros].mask(df[zeros] == 0)
for b in real_zero:  # a 0 standard with neither limit set is a placeholder too
    unset = df[b + "_min"].isna() & df[b + "_max"].isna() & (df[b + "_estandar"] == 0)
    df.loc[unset, b + "_estandar"] = np.nan
print("values turned into NaN:")
(df.isna().sum() - na_before).loc[lambda s: s > 0]

In [ ]:
# 8. Missing values: gaps are mostly structural (greases vs oils), so compare by family.
#    Drop columns with fewer than MIN_DATOS values (too few to compare products on).
miss = df.drop(columns="familia").isna().groupby(df["familia"]).mean().T.mul(100).round(0)
miss["total"] = df.isna().mean().mul(100).round(0)
miss = miss.sort_values("total", ascending=False)
miss.head(25).plot.barh(figsize=(8, 7)).invert_yaxis()
plt.xlabel("% missing")
plt.show()

n_filled = df.notna().sum()
few = n_filled.index[n_filled < MIN_DATOS]
df = df.drop(columns=few)
print(f"dropped (< {MIN_DATOS} values):", n_filled[few].to_dict(), "| shape:", df.shape)

# sparse but kept: decide at modeling time
miss.loc[miss["total"] > 90].drop(index=few).assign(n_filled=df.notna().sum())

In [ ]:
# 9. Irrelevant columns: constants (deteccion_de_grumo_valor only ever says "sin grumo"; whether it is filled just mirrors
#    familia) and grado_iso_vg_valor, a copy of grado_iso_vg (same value wherever both exist, and never filled alone)
const = [c for c in df.columns if df[c].nunique() <= 1]
both = df["grado_iso_vg"].notna() & df["grado_iso_vg_valor"].notna()
assert (df.loc[both, "grado_iso_vg"] == df.loc[both, "grado_iso_vg_valor"]).all()
assert not (df["grado_iso_vg_valor"].notna() & df["grado_iso_vg"].isna()).any()
irrelevant = const + ["grado_iso_vg_valor"]
print("dropped:", {c: df[c].dropna().unique().tolist()[:3] for c in irrelevant})
df = df.drop(columns=irrelevant)
print("shape:", df.shape)

In [ ]:
# 10. Data casting: registro_nsf is a registration ID, not a measure (5 pairs of different products share one),
#     so keep only whether the product has it; copper corrosion class is ordinal; the rest of the text is nominal.
df.insert(df.columns.get_loc("registro_nsf"), "tiene_registro_nsf", df["registro_nsf"].notna())
df = df.drop(columns="registro_nsf")
df["corrosion_al_cobre"] = pd.Categorical(df["corrosion_al_cobre"], categories=["1a", "1b"], ordered=True)
cat_cols = df.select_dtypes(exclude=["number", "bool", "boolean", "category"]).columns.drop("codigoproducto")
df[cat_cols] = df[cat_cols].astype("category")
print(df.dtypes.astype(str).value_counts())
print("NSF:", df["tiene_registro_nsf"].sum(), "products")

In [ ]:
# 11. NLGI: the declared grade has no 00/000 (they are coded as 0). Derive the grade from cone penetration at 25 °C
#     (NLGI ranges, nearest center; 00 -> -1, 000 -> -2) so fluid greases can be told apart. The declared grade is kept.
nlgi_ranges = {-2: (445, 475), -1: (400, 430), 0: (355, 385), 1: (310, 340), 2: (265, 295),
               3: (220, 250), 4: (175, 205), 5: (130, 160), 6: (85, 115)}
centers = pd.Series({g: (lo + hi) / 2 for g, (lo, hi) in nlgi_ranges.items()})
pen = df["penetracion_de_cono_a_25c_0_1mm"].where(df["familia"] == "grasa")
derived = pen.map(lambda x: centers.sub(x).abs().idxmin() if pd.notna(x) else np.nan)
df.insert(df.columns.get_loc("grado_nlgi_consistencia") + 1, "grado_nlgi_por_penetracion", derived)
pd.crosstab(df["grado_nlgi_consistencia"], df["grado_nlgi_por_penetracion"])

In [ ]:
# 12. Consistency. 12a: min <= estandar <= max per property; negatives should only appear in temperatures
#     (and in grado_nlgi_por_penetracion, where 00 = -1 and 000 = -2 by design)
bases = [c[:-4] for c in df.columns if c.endswith("_min") and c[:-4] + "_max" in df]
checks = pd.DataFrame(
    {
        b: {
            "min>max": (df[b + "_min"] > df[b + "_max"]).sum(),
            "estandar_out_of_range": (
                (df[b + "_estandar"] < df[b + "_min"]) | (df[b + "_estandar"] > df[b + "_max"])
            ).sum() if b + "_estandar" in df else np.nan,
        }
        for b in bases
    }
).T
print(checks[checks.sum(axis=1) > 0])
num = df[[c for c in df.columns if pd.api.types.is_float_dtype(df[c])]]
neg = (num < 0).sum()
neg[neg > 0]

In [ ]:
# 12b. Contradictions between columns: listed for Interlub, not fixed (we cannot know which side is right)
g = df["grado_nlgi_consistencia"]
same_color = {"crema": "beige", "plata": "gris"}  # synonyms, not contradictions
first_word = lambda s: s.astype("str").str.split().str[0].replace(same_color).where(s.notna())
rules = {
    "nlgi declarado != nlgi_valor (control de calidad)":
        g.notna() & df["grado_nlgi_valor"].notna() & (g != df["grado_nlgi_valor"]),
    "nlgi declarado != nlgi por penetracion":  # 0 vs 00/000 is the known coding issue, not a contradiction
        ((g - df["grado_nlgi_por_penetracion"]).abs() >= 1) & ~((g == 0) & (df["grado_nlgi_por_penetracion"] < 0)),
    "grado iso vg fuera de la especificacion de viscosidad a 40 c":
        (df["grado_iso_vg"] < df["viscosidad_40_c_min"]) | (df["grado_iso_vg"] > df["viscosidad_40_c_max"]),
    "color != color_visual_valor":
        first_word(df["color"]).notna() & first_word(df["color_visual_valor"]).notna()
        & (first_word(df["color"]) != first_word(df["color_visual_valor"])),
    "aceite con propiedades de grasa (textura, grumo o penetracion)": aceite_tipo_grasa,
    "punto de gota < temperatura maxima de servicio": df["punto_de_gota_c"] < df["temperatura_de_servicio_c_max"],
    "punto de flama < temperatura maxima de servicio": df["punto_de_flama_c"] < df["temperatura_de_servicio_c_max"],
    "punto de congelamiento > temperatura minima de servicio":
        df["punto_de_congelamiento_c"] > df["temperatura_de_servicio_c_min"],
}
review = pd.concat(
    [pd.DataFrame({"codigoproducto": df.loc[m.fillna(False), "codigoproducto"], "problema": name}) for name, m in rules.items()]
    + [pd.DataFrame(not_numeric, columns=["columna", "codigoproducto", "valor"]).assign(problema="valor no numerico (-> NaN)")],
    ignore_index=True,
)
print(review["problema"].value_counts())
review

In [ ]:
# 13. Outliers: IQR within each familia, on log10 for positive columns (viscosities span orders of magnitude).
#     Reported, not removed: extreme products (e.g. 1000-1400 °C bentonite pastes) must still be recommendable.
def iqr_outliers(s):
    x = np.log10(s) if s.min() > 0 else s
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    return (x < q1 - 1.5 * (q3 - q1)) | (x > q3 + 1.5 * (q3 - q1))


out = num.groupby(df["familia"], observed=True).transform(iqr_outliers)
n_out = out.sum().sort_values(ascending=False)
print(n_out.head(12))

fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, c in zip(axes.ravel(), n_out.index[:12]):
    sns.boxplot(data=df, x="familia", y=c, ax=ax, log_scale=bool(num[c].min() > 0))
    ax.set_title(c, fontsize=8)
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# 14. Export (processed/*.csv is git-ignored; encrypt and track with the DVC scripts to share it).
#     CSV does not keep dtypes: re-cast categories/booleans after loading if needed.
df.to_csv(OUT, index=False)
review.to_csv(REVIEW, index=False)
print("saved:", df.shape, "| read back:", pd.read_csv(OUT).shape)
print("review:", review.shape, "|", review["codigoproducto"].nunique(), "products")